# Circuit registration

Copyright (c) 2026 Open Brain Institute

Authors: Christoph Pokorny

Last modified: 06.2026

## Summary
This notebook allows a user to register a new SONATA circuit together with metadata as a private `Circuit` entity in the entitycore database. Additional circuit assets (connectivity matrix, basic connectivity plots, etc.) will be generated automatically upon upload.

## Use
The SONATA circuit should be provided by compressing the whole circuit folder into a zip file and copying it into the JupyterHub workspace. All required metadata can be entered diretly within the notebook. Where applicable, interactive widgets can be used to select metadata options and linked entities in a user-friendly way. All required linked entities (publications, contributors, subjects, etc.) must exist already in the database, i.e., they have to be registered separately beforehand.

Optionally, the main overview and simulation designer images can be provided as well; otherwise default images will be generated.

**Important:** [SNAP circuit validation](https://github.com/openbraininstitute/snap#circuit-validation) needs to pass for a new circuit to be registered.


In [ ]:
from entitysdk import Client
from obi_auth import get_token
from obi_notebook import get_projects
from obi_notebook.get_environment import get_environment
from obi_one.utils.circuit_registration import register_circuit_from_metadata
from helpers import assemble_circuit_metadata, assemble_contributions, assemble_publications, check_registered_circuit, create_contributor_widgets, create_metadata_widgets, create_publication_widgets, query_metadata_options, validate_paths

In [ ]:
# Disable info logging (optional)
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)

## Project selection

As a first step, we select the project under which to register the new circuit as a private `Circuit` entity.


In [ ]:
# environment = get_environment()
environment = "staging"
token = get_token(environment=environment, auth_mode="daf")
project_context = get_projects.get_projects(token, env=environment)

Then, we get the database client for accessing entitycore.

In [ ]:
client = Client(environment=environment, project_context=project_context, token_manager=token)

## Circuit information

### General metadata

**IMPORTANT:**
- **The circuit name must not yet exist in entitycore (to avoid duplicates)**

- **Linked entities must exist in entitycore. Otherwise, they must first be registered:**
  - Species
  - Subject
  - Brain region hierarchy
  - Brain region
  - License


In [ ]:
# Get all metadata options
metadata_options = query_metadata_options(client)

In [ ]:
# Display metadata widgets
w = create_metadata_widgets(options)

In [ ]:
# Validate inputs and assemble metadata dict
circuit_metadata = assemble_circuit_metadata(w, options)

### Circuit and image paths

Path to the compressed SONATA circuit in `.gz` format.

In [ ]:
# Compressed SONATA circuit folder with circuit_config.json
circuit_path = "./circuit.gz"

Optionally, paths to pre-existing overview and simulation designer images can be provided. Otherwise, they will be generated automatically.

In [ ]:
# Optional: Pre-existing image files (skip generation if provided)
overview_image_path = None  # Path to overview image (.png or .webp)
sim_designer_image_path = None  # Path to simulation designer image (.png)

In [ ]:
# Validate paths
validate_paths(circuit_path, overview_image_path, sim_designer_image_path)

### Optional: Contributors

Hold **Cmd** (Mac) or **Ctrl** (Windows/Linux) to select multiple entries. Note that all contributors will be registered without specifying their role.

**IMPORTANT:**

- **Contributors must already exist in entitycore. Otherwise, they must first be registered.**

In [ ]:
# Display contributor widgets
contrib_widgets = create_contributor_widgets(metadata_options)

In [ ]:
# Assemble contributor dict
circuit_contributions = assemble_contributions(contrib_widgets)

### Optional: Publications

Hold **Cmd** (Mac) or **Ctrl** (Windows/Linux) to select multiple entries.

**IMPORTANT:**

- **Publications must already exist in entitycore. Otherwise, they must first be registered.**

In [ ]:
# Display publication widgets
pub_widgets = create_publication_widgets(metadata_options)

In [ ]:
# Assemble publication dict
circuit_publications = assemble_publications(pub_widgets)

## Run circuit registration

First, a dry run (`dry_run = True`) should be used to check the circuit files, metadata, and dependencies but w/o registration to entitycore:
- Valid SONATA circuit with circuit_config.json must exist
- Additional assets will be generated locally
- The circuit name must not yet exist in entitycore (i.e., to avoid duplicates)
- All dependencies must already exist in entitycore (i.e., must be registered beforehand)

If successful, the actual registration can be done by re-running with `dry_run = False`.

In [70]:
dry_run = True  # If True, runs all checks but no actual registration; otherwise actual registration

In [ ]:
# Create & register circuit entity
registered_circuit = register_circuit_from_metadata(
    client=client,
    circuit_metadata=circuit_metadata,
    circuit_path=circuit_path,
    contributions=circuit_contributions,
    publications=circuit_publications,
    overview_image_path=overview_image_path,
    sim_designer_image_path=sim_designer_image_path,
    dry_run=dry_run,
)

After registration, the registered Circuit entity can be checked.

In [ ]:
# Check registered circuit
check_registered_circuit(client, registered_circuit)